In [ ]:
import torch
import numpy as np

from utils import *

d1 = 10000
d2 = 1000
r = 10
p = 0.1
ob = 2
sample = "iid"
if torch.cuda.is_available():
    device = 'cuda:0'
else:
    device = 'cpu'

# dataset
dataset = "syn"
#print(dataset)

total_t = 5
total_T_var_list = []
total_T_p_var_list = []
total_T_err_list = []
total_T_p_err_list = []
for t in range(total_t):
    M = load_data_syn(r, d1, d2, device)
    
    runs = 100
    T_p_list = []
    T_list = []
    T_p_err_list = []
    T_err_list = []
    for run in range(runs):

        if sample == "iid":
            # IID sample
            observed_M, masks = get_uniform_masks(M, p)
        else:
            # few entry sample
            observed_M, masks = get_random_samples_per_row(M.cpu().numpy(), ob)
            p = ob/d2
            observed_M = torch.from_numpy(observed_M).float().to(device)
            masks = torch.from_numpy(masks).to(device)

        # observed second-moment matrix
        MTM = M.T @ M
        cov_observe_M =  observed_M.T @ observed_M
        T_masks = 1 * (cov_observe_M!=0)

        # True probability weighting
        diag_cov = torch.diag( torch.diag(cov_observe_M) )
        T_p = ((1.0 / p) * diag_cov + (1.0 / (p**2)) * (cov_observe_M - diag_cov))
        #T_p = normalize_matrix(T_p)
        T_p_err_list.append((torch.norm(T_p*T_masks - MTM*T_masks)/torch.norm(MTM*T_masks)).item())
        T_p = T_p / torch.mean(T_p)
        T_p_list.append(T_p.unsqueeze(0))
        #print("relative error of T_p: ", torch.norm(T_p*T_masks - MTM*T_masks)/torch.norm(MTM*T_masks))

        # Inverse estimated probability weighting in matrix
        cov_observe_count = (1 * (observed_M != 0)).float().T @ (1 * (observed_M != 0).float())
        cov_observe_count = cov_observe_count + (cov_observe_count == 0) * 1
        p_hat_matrix = cov_observe_count/d1

        T = cov_observe_M / p_hat_matrix
        T_err_list.append((torch.norm(T*T_masks - MTM*T_masks)/torch.norm(MTM*T_masks)).item())
        #T = normalize_matrix(T)
        T = T / torch.mean(T)
        T_list.append(T.unsqueeze(0))

        # calculate matrix of p_hat and p
        p_hat_matrix = (cov_observe_count/d1)
        p_matrix = (torch.diag(torch.diag(torch.ones(d2, d2)*(p-p**2))) + torch.ones(d2, d2)*p**2).to(device)

        # calculate (p_hat - p)
        #print("mean value of (p_hat - p) in matrix: ", (p_matrix*T_masks-p_hat_matrix*T_masks).mean()/(T_masks.sum() / d2**2))
        # calculate the relative err of T_p_hat
        #print("relative error of T_p_hat: ", torch.norm(T*T_masks-MTM*T_masks)/torch.norm(MTM*T_masks))
    #print("relative error of T_p: ", torch.norm(T_p*T_masks - MTM*T_masks)/torch.norm(MTM*T_masks))
    #print("relative error of T: ", torch.norm(T*T_masks - MTM*T_masks)/torch.norm(MTM*T_masks))
    total_T_err_list.append(np.mean(T_err_list))
    total_T_p_err_list.append(np.mean(T_p_err_list))

    mean_T_p = torch.cat(T_p_list).mean(0)
    mean_T = torch.cat(T_list).mean(0)
    var_T_p = torch.cat(T_p_list).var(0)
    var_T = torch.cat(T_list).var(0)

    var_T_p_value = (var_T_p).mean()
    var_T_value = (var_T).mean()

    diag_var_T_p = torch.diag(torch.diag(var_T_p))
    offdiag_var_T_p = var_T_p - diag_var_T_p
    diag_var_T = torch.diag(torch.diag(var_T))
    offdiag_var_T = var_T - diag_var_T

    num_diag = d2
    num_off = d2*(d2-1)/2

    total_T_p_var_list.append(var_T_p_value.item())
    total_T_var_list.append(var_T_value.item())

    #print("var of T_p: ", var_T_p_value)
    #print("var of T: ", var_T_value)

    #print("var of T_p in diag: ", diag_var_T_p.sum())
    #print("var of T in diag: ", diag_var_T.sum())

    #print("var of T_p in off-diag: ", offdiag_var_T_p.sum())
    #print("var of T in off-diag: ", offdiag_var_T.sum())

print(f"mean of err of T_p: {np.mean(total_T_p_err_list)} +- {np.std(total_T_p_err_list)} ")
print(f"mean of err of T: {np.mean(total_T_err_list)} +- {np.std(total_T_err_list)} ")

print(f"mean of var of T_p: {np.mean(total_T_p_var_list)} +- {np.std(total_T_p_var_list)} ")
print(f"mean of var of T: {np.mean(total_T_var_list)} +- {np.std(total_T_var_list)} ")
